# Spotify Data Cleaning: Weekly Albums

* Simple pandas cleaning for `charts_albums_weekly_52W.csv`. 
* The final DataFrame is `clean_albums_weekly_52W`. 
* No intermediate CSV export. 
* The last cell loads into MySQL staging.
* Run cells in order. Original CSV goes in `data/raw/`.

---

## 1. Import and read source

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# Find the source file from the project folder.
def locate_raw_data(filename):
    for folder in (Path.cwd(), *Path.cwd().parents):
        candidate = folder / "data" / "raw" / filename
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Missing CSV in data/raw/: {filename}")

albums_weekly_df = pd.read_csv(locate_raw_data("charts_albums_weekly_52W.csv"))
print("Source rows:", len(albums_weekly_df))

Source rows: 526800


---

## 2. Standardize source columns

In [3]:
# Preserve entry/peak history, artist names and multi-artist relationships.
albums_weekly_df = albums_weekly_df.rename(columns={"date": "week_start_date", "uri": "album_uri"})

# Convert the weekly chart and historical dates.
for column in ["week_start_date", "entry_date", "peak_date", "release_date"]:
    if column in albums_weekly_df:
        albums_weekly_df[column] = pd.to_datetime(albums_weekly_df[column], errors="coerce")

# Standardize strings without changing proper-name capitalization.
for column in ["country", "album_name", "label", "artist_names", "entry_status"]:
    if column in albums_weekly_df:
        albums_weekly_df[column] = albums_weekly_df[column].astype("string").str.strip()
if "country" in albums_weekly_df:
    albums_weekly_df["country"] = albums_weekly_df["country"].str.upper()
if "entry_status" in albums_weekly_df:
    albums_weekly_df["entry_status"] = albums_weekly_df["entry_status"].str.replace("_", " ", regex=False)

# Preserve the source capitalization of album names and labels.
albums_weekly_df["album_uri"] = (albums_weekly_df["album_uri"].astype("string")
    .str.strip().str.replace(r"^spotify:album:", "", regex=True))
# Retain multiple artist IDs for later relationship mapping.
if "artist_uris" in albums_weekly_df:
    albums_weekly_df["artist_uris"] = (albums_weekly_df["artist_uris"].astype("string")
        .str.strip().str.replace("spotify:artist:", "", regex=False))

# Do not substitute missing release dates with another album's release date.
# Convert the available performance metrics to nullable integers.
for column in ["entry_rank", "peak_rank", "rank", "previous_rank", "weeks_on_chart", "streams"]:
    if column in albums_weekly_df:
        albums_weekly_df[column] = pd.to_numeric(albums_weekly_df[column], errors="coerce").astype("Int64")


---

## 3. Handle missing values and duplicates

In [4]:
# One observation per chart key; artist master uses one row per artist URI.
keys = ["week_start_date", "country", "album_uri"]
required = ["week_start_date", "country", "album_uri", "rank"]
expected_columns = ['week_start_date', 'country', 'rank', 'album_uri', 'album_name', 'artist_names', 'label', 'peak_rank', 'previous_rank', 'weeks_on_chart', 'consecutive_weeks', 'entry_status', 'peak_date', 'entry_rank', 'entry_date', 'release_date', 'artist_uris']
missing_columns = set(expected_columns) - set(albums_weekly_df.columns)
if missing_columns:
    raise ValueError(f"Missing staging columns: {sorted(missing_columns)}")

# Standardize blank text. Keep optional unknown measures as SQL NULL.
for col in albums_weekly_df.select_dtypes(include=["object", "string"]).columns:
    albums_weekly_df[col] = albums_weekly_df[col].astype("string").str.strip().replace("", pd.NA)

# Descriptive labels have meaningful defaults; unknown statistics do not.
for col, default in {'album_name': 'Unknown Album', 'artist_names': 'Unknown Artist', 'label': 'Unknown Label'}.items():
    albums_weekly_df[col] = albums_weekly_df[col].fillna(default)

# Invalid optional ranks/counts are unknown, not zero.
for col in ['peak_rank', 'entry_rank', 'previous_rank']:
    values = pd.to_numeric(albums_weekly_df[col], errors="coerce")
    albums_weekly_df[col] = values.where(values > 0).astype("Int64")
for col in ['weeks_on_chart', 'consecutive_weeks']:
    values = pd.to_numeric(albums_weekly_df[col], errors="coerce")
    albums_weekly_df[col] = values.where(values >= 0).astype("Int64")

# Required chart keys cannot be invented; keep only valid observations.
valid = albums_weekly_df[required].notna().all(axis=1) & albums_weekly_df["rank"].between(1, 200).fillna(False)

# Keep optional chart attributes within the staging constraints.
for col in ["peak_rank", "entry_rank"]:
    albums_weekly_df[col] = albums_weekly_df[col].where(albums_weekly_df[col].between(1, 200))
albums_weekly_df["consecutive_weeks"] = albums_weekly_df["consecutive_weeks"].where(
    albums_weekly_df["weeks_on_chart"].isna() | (albums_weekly_df["consecutive_weeks"] <= albums_weekly_df["weeks_on_chart"])
)

invalid_count = int((~valid).sum())
albums_weekly_df = albums_weekly_df.loc[valid]

# Fast single-pass grain deduplication. Keep first original source row.
duplicate_count = int(albums_weekly_df.duplicated(subset=keys).sum())
clean_albums_weekly_52W = albums_weekly_df.drop_duplicates(subset=keys, keep="first").reset_index(drop=True)

print("Invalid key/rank rows removed:", invalid_count)
print("Repeated business keys removed:", duplicate_count)
print("Final staging rows:", len(clean_albums_weekly_52W))


Invalid key/rank rows removed: 0
Repeated business keys removed: 0
Final staging rows: 526800


---

## 4. Review and load to MySQL

In [5]:
print("DataFrame:", "clean_albums_weekly_52W")
print("Rows:", len(clean_albums_weekly_52W))
print("Missing values (optional columns):")
print(clean_albums_weekly_52W.isna().sum().to_string())
display(clean_albums_weekly_52W.head(5))

DataFrame: clean_albums_weekly_52W
Rows: 526800
Missing values (optional columns):
week_start_date          0
country                  0
rank                     0
album_uri                0
album_name               0
artist_names             0
label                    0
peak_rank                0
previous_rank        30942
weeks_on_chart           0
consecutive_weeks        0
entry_status             0
peak_date                0
entry_rank               0
entry_date               0
release_date         17354
artist_uris              0


,week_start_date,country,rank,album_uri,album_name,artist_names,label,peak_rank,previous_rank,weeks_on_chart,consecutive_weeks,entry_status,peak_date,entry_rank,entry_date,release_date,artist_uris
0,2025-11-06,AR,1,0sQR1p7NyAUqMPmWdZ6UBd,La Vida Era Más Corta,Milo j,Sony Music Latin,1,1,6,6,NO CHANGE,2025-10-02,1,2025-10-02,2025-09-25,19HM5j0ULGSmEoRcrSe5x3
1,2025-11-06,AR,2,5K79FLRUCSysQnVESLcTdb,DeBÍ TiRAR MáS FOToS,Bad Bunny,Rimas Entertainment LLC.,1,2,44,44,NO CHANGE,2025-01-09,1,2025-01-09,2025-01-05,4q3ewBCX7sLwd24euuV69X
2,2025-11-06,AR,3,4xMNn5AFdmBIYiXRbt5YH4,BORONDO,Beéle,5020 Records,2,3,25,25,NO CHANGE,2025-06-19,13,2025-05-22,2025-05-15,7a0XAaPaK2aDSqa8p3QnC7
3,2025-11-06,AR,4,13P9Qaty5SUmKgpSvpbOYt,El Regreso,Tan Bionica,DALE PLAY Records,4,<NA>,1,1,NEW ENTRY,2025-11-06,4,2025-11-06,2025-11-04,37MCoi4pcUf9EKsPXeuCqU
4,2025-11-06,AR,5,3RQQmkQEvNCY4prGKE6oc5,Un Verano Sin Ti,Bad Bunny,Rimas Entertainment LLC,1,5,183,183,NO CHANGE,2022-05-12,1,2022-05-12,2022-05-06,4q3ewBCX7sLwd24euuV69X


In [6]:
# Import the shared staging loader from etl/load.py.
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "etl" / "load.py").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Missing etl/load.py in the project folder")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from etl.load import load_dataframe

In [7]:
# Load only the cleaned DataFrame; no exclusion-triggered ValueError.
if clean_albums_weekly_52W.empty:
    raise ValueError("No valid records available for staging.")
load_dataframe(clean_albums_weekly_52W, "clean_albums_weekly_52W")
print("Loaded:", "clean_albums_weekly_52W", "| Rows:", len(clean_albums_weekly_52W))

Loaded 526,800 rows into spotify_analytics_stg.clean_albums_weekly_52W
Loaded: clean_albums_weekly_52W | Rows: 526800
